# Trolley Problem LLM Bias Study

Notebook sottile di orchestrazione: tutta la logica vive in `src/trolley/`, qui si
chiamano solo le funzioni nell'ordine giusto, con markdown esplicativo tra un passo e
l'altro. Prima di lanciare il run completo, prova con una configurazione ridotta
(vedi commenti nella cella di setup) per validare l'intera pipeline in pochi minuti.


In [3]:
import sys
from pathlib import Path

# Rende importabile src/trolley/ eseguendo il notebook da notebooks/
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from trolley.config import load_config
from trolley.cache import ResponseCache
from trolley.providers.local_hf import LocalHFProvider
from trolley.providers.gemini import GeminiProvider
from trolley import scenario_generator, pipeline, analysis, plots, report

config = load_config(str(PROJECT_ROOT / "config" / "config.yaml"))
print(f"Scenari: {config.n_scenarios_per_trait} x {len(config.traits_of_interest)} tratti "
      f"+ {config.n_control_scenarios} controllo | Ripetizioni: {config.repetitions}")
print(f"Modelli locali: {[m.label for m in config.local_models]} | Gemini: {config.gemini_model}")


ModuleNotFoundError: No module named 'google'

## 1. Generazione degli scenari

Genera (con seed fisso, quindi riproducibile) gli scenari a tratto controllato e quelli
di controllo puramente numerico, e li salva in `scenarios/generated_scenarios.json`.


In [ ]:
scenarios = scenario_generator.generate_and_save(config)
print(f"Generati {len(scenarios)} scenari")

families = {}
for s in scenarios:
    families[s["family"]] = families.get(s["family"], 0) + 1
for family, count in sorted(families.items()):
    print(f"  {family}: {count}")

print("\nEsempio (scenario a tratto controllato):")
example = next(s for s in scenarios if s["varied_trait"])
print(example["description"])


## 2. Esperimento - modelli locali

Ogni modello viene caricato, eseguito su tutti gli scenari x ripetizioni configurate, e
scaricato prima di passare al successivo (per liberare VRAM). Grazie alla cache su disco
(`results/cache/responses.jsonl`), rilanciare questa cella dopo un'interruzione riprende
da dove si era interrotto invece di rigenerare tutto.

**Nota sui tempi**: con la configurazione di default (50+ scenari, 5 ripetizioni, 3
modelli) questo passo richiede diverse ore. Per un primo test rapido, riduci
`repetitions` e/o `local_models` in `config/config.yaml` prima di eseguire.


In [ ]:
cache = ResponseCache(config.paths.cache_dir)

for model_cfg in config.local_models:
    print(f"\n=== {model_cfg.label} ({model_cfg.id}) ===")
    provider = LocalHFProvider(model_cfg.id, model_cfg.label, hf_token=config.hf_token)
    try:
        pipeline.run_for_provider(provider, scenarios, config, cache)
    finally:
        provider.unload()


## 3. Esperimento - Gemini

Stessa logica, stessa cache, ma via API: nessuna estrazione di attenzione (modello
black-box). Rispetta il rate limit della tua chiave/tier configurando eventuali pause se
necessario.


In [ ]:
gemini_provider = GeminiProvider(config.gemini_model, label="gemini", api_key=config.gemini_api_key)
pipeline.run_for_provider(gemini_provider, scenarios, config, cache)


## 4. Analisi

Costruisce il dataset piatto dalla cache, calcola la colonna `sacrifice_target` (la
persona col valore "target" del tratto e' stata sacrificata?), esegue le regressioni sui
bias per tratto/modello e i test statistici di confronto tra modelli.


In [ ]:
df = analysis.build_flat_dataframe(cache)
df = analysis.add_target_attention(df)
analysis.save_flat_csv(df, config.paths.responses_flat_csv)
print(f"Righe totali: {len(df)}")

bias_df = analysis.run_bias_regressions(df)
bias_df.to_csv(config.paths.bias_regression_csv, index=False)
bias_df


In [ ]:
cross_model = analysis.run_cross_model_tests(df)
cross_model


## 5. Grafici

Le figure vengono salvate in `results/figures/` e riusate nel report finale.


In [ ]:
plots.plot_decision_distribution(df, config.paths.figures_dir)
plots.plot_framework_distribution(df, config.paths.figures_dir)
plots.plot_bias_regression(bias_df, config.paths.figures_dir)
plots.plot_attention_vs_bias(df, config.paths.figures_dir)
print(f"Figure salvate in: {config.paths.figures_dir}")


## 6. Report finale

Assembla `report/report.md` (introduzione, metodologia, risultati, limiti, conclusioni)
a partire dai dati e dalle figure appena generati. Per un PDF: `pandoc report/report.md -o report/report.pdf` (documentato nel README).


In [ ]:
import datetime

report_text = report.generate_report(
    df, bias_df, cross_model,
    figures_dir=config.paths.figures_dir,
    out_path=config.paths.report_path,
    generated_at=datetime.datetime.now().strftime("%d/%m/%Y %H:%M"),
)
print(report_text[:1500])
